                    RAG SYSTEM

Question
   │
   ▼
Embedding Model
   │
   ▼
Retriever: Get chucks of ducments suitable to query
   │
   ▼
Top-K Chunks
   │
   ▼
Prompt Builder
   │
   ▼
Generator: Use LLM api  to get answers from the prompt
   │
   ▼
Answer

In [2]:
import sys
from pathlib import Path

project_root = Path.cwd().parent
sys.path.append(str(project_root))

In [3]:
from src.config import RAW_DOCUMENTS_DIR, TOP_K
from src.loader import load_documents
from src.pipeline import build_knowledge_base
from src.retriever import retrieve
from src.rag import create_rag_prompt

/home/prashant/Projects/rag-knowledge-assistant/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
documents = load_documents(
    RAW_DOCUMENTS_DIR
)

print(f"Loaded {len(documents)} documents.")

for document in documents:
    print(document["source"])

Loaded 1 documents.
company_policy.txt


In [7]:
model, index, chunks = build_knowledge_base(
    documents
)

print(f"Created {len(chunks)} chunks.")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2432.88it/s]


Created 1 chunks.


In [ ]:
print(chunks[0])

{'text': "Company Annual Leave Policy Employees receive 30 days of annual leave per calendar year. Employees should request annual leave through the company's HR system. Unused annual leave may be carried over to the following year, subject to company policy. Employees should normally request leave at least two weeks in advance. Remote Work Policy Employees may work remotely up to three days per week with manager approval. Employees working remotely must remain available during their agreed working hours.", 'source': 'company_policy.txt'}


In [8]:
question = "How many days of annual leave do employees receive?"

print(question)

How many days of annual leave do employees receive?


In [9]:
retrieved_chunks = retrieve(
    query=question,
    model=model,
    index=index,
    chunks=chunks,
    top_k=TOP_K,
)

In [10]:
prompt = create_rag_prompt(
    question=question,
    retrieved_chunks=retrieved_chunks,
)

print(prompt)

You are a corporate knowledge assistant.

Answer the user's question using only the provided context.

Rules:
1. Do not invent information.
2. If the answer cannot be found in the context, say:
   "The information was not found in the provided documents."
3. Cite the source that supports your answer.
4. If multiple sources support the answer, cite each relevant source.
5. Keep the answer concise and factual.

Context:
[Source 1: company_policy.txt]
Company Annual Leave Policy Employees receive 30 days of annual leave per calendar year. Employees should request annual leave through the company's HR system. Unused annual leave may be carried over to the following year, subject to company policy. Employees should normally request leave at least two weeks in advance. Remote Work Policy Employees may work remotely up to three days per week with manager approval. Employees working remotely must remain available during their agreed working hours.

Question:
How many days of annual leave do em

In [11]:
print("QUESTION")
print(question)

print("\nRETRIEVED CONTEXT")
for result in retrieved_chunks:
    print(result["text"])

print("\nRAG PROMPT")
print(prompt)

QUESTION
How many days of annual leave do employees receive?

RETRIEVED CONTEXT
Company Annual Leave Policy Employees receive 30 days of annual leave per calendar year. Employees should request annual leave through the company's HR system. Unused annual leave may be carried over to the following year, subject to company policy. Employees should normally request leave at least two weeks in advance. Remote Work Policy Employees may work remotely up to three days per week with manager approval. Employees working remotely must remain available during their agreed working hours.

RAG PROMPT
You are a corporate knowledge assistant.

Answer the user's question using only the provided context.

Rules:
1. Do not invent information.
2. If the answer cannot be found in the context, say:
   "The information was not found in the provided documents."
3. Cite the source that supports your answer.
4. If multiple sources support the answer, cite each relevant source.
5. Keep the answer concise and fact

In [12]:
from src.generator import generate_answer

answer = generate_answer(prompt)

print(answer)

The RAG prompt has been successfully created. A real language model needs to be connected to generate the final answer.


In [13]:
question = "How many days of annual leave do employees receive?"

retrieved_chunks = retrieve(
    query=question,
    model=model,
    index=index,
    chunks=chunks,
    top_k=TOP_K,
)

prompt = create_rag_prompt(
    question=question,
    retrieved_chunks=retrieved_chunks,
)

print(prompt)

You are a corporate knowledge assistant.

Answer the user's question using only the provided context.

Rules:
1. Do not invent information.
2. If the answer cannot be found in the context, say:
   "The information was not found in the provided documents."
3. Cite the source that supports your answer.
4. If multiple sources support the answer, cite each relevant source.
5. Keep the answer concise and factual.

Context:
[Source 1: company_policy.txt]
Company Annual Leave Policy Employees receive 30 days of annual leave per calendar year. Employees should request annual leave through the company's HR system. Unused annual leave may be carried over to the following year, subject to company policy. Employees should normally request leave at least two weeks in advance. Remote Work Policy Employees may work remotely up to three days per week with manager approval. Employees working remotely must remain available during their agreed working hours.

Question:
How many days of annual leave do em

In [14]:
question = "How many days of annual leave do employees receive?"

results = retrieve(
    query=question,
    model=model,
    index=index,
    chunks=chunks,
    top_k=3,
)

for result in results:
    print(
        f"Score: {result['score']:.3f}"
    )
    print(result["text"])
    print("-" * 60)

Score: 0.728
Company Annual Leave Policy Employees receive 30 days of annual leave per calendar year. Employees should request annual leave through the company's HR system. Unused annual leave may be carried over to the following year, subject to company policy. Employees should normally request leave at least two weeks in advance. Remote Work Policy Employees may work remotely up to three days per week with manager approval. Employees working remotely must remain available during their agreed working hours.
------------------------------------------------------------


In [15]:
question = "What is the company's policy on purchasing airplanes?"

results = retrieve(
    query=question,
    model=model,
    index=index,
    chunks=chunks,
    top_k=3,
)

print(f"Retrieved chunks: {len(results)}")

for result in results:
    print(result["score"])
    print(result["text"])

Retrieved chunks: 1
0.17571280896663666
Company Annual Leave Policy Employees receive 30 days of annual leave per calendar year. Employees should request annual leave through the company's HR system. Unused annual leave may be carried over to the following year, subject to company policy. Employees should normally request leave at least two weeks in advance. Remote Work Policy Employees may work remotely up to three days per week with manager approval. Employees working remotely must remain available during their agreed working hours.


In [ ]:
for chunk in chunks:
    print(
        f"Chunk ID: {chunk['chunk_id']} "
        f"| Source: {chunk['source']}"
    )
    print(chunk["text"])
    print("-" * 60)